In [1]:
import litellm
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("OPENAI_API_KEY"):
    litellm.openai_key = os.getenv("OPENAI_API_KEY")

OPENAI_MODEL = os.getenv("OPENAI_MODEL")

In [2]:
demo_dataset = [
    {"review": "An absolute masterpiece, the best film of the year!", "label": 1},  # 0
    {
        "review": "I was bored from start to finish. A total waste of time.",
        "label": 0,
    },  # 1
    {
        "review": "The acting was incredible and the story was so moving.",
        "label": 1,
    },  # 2
    {"review": "While not perfect, it had some good moments.", "label": 1},  # 3
    {"review": "A confusing plot and terrible dialogue.", "label": 0},  # 4
]

print(f"Dataset created with {len(demo_dataset)} reviews.")

Dataset created with 5 reviews.


In [3]:
id2label = {
    0: "NEGATIVE", 1: "POSITIVE",
}

for i in range(2):
    review = demo_dataset[i]["review"]
    label_id = demo_dataset[i]["label"]
    print(f"label={id2label[label_id]}, review={review}")

label=POSITIVE, review=An absolute masterpiece, the best film of the year!
label=NEGATIVE, review=I was bored from start to finish. A total waste of time.


In [4]:
reviews_string = ""

for i, entry in enumerate(demo_dataset):
    reviews_string += f"{i} -> {entry['review']}\n"

SYSTEM_PROMPT = """You are a helpful assistant that classifies movie reviews as POSITIVE or NEGATIVE.
Respond only with a JSON object where the keys are the review numbers and the values are the classification."""

USER_PROMPT = reviews_string

print("SYSTEM PROMPT")
print(SYSTEM_PROMPT)
print("USER PROMPT")
print(USER_PROMPT)


SYSTEM PROMPT
You are a helpful assistant that classifies movie reviews as POSITIVE or NEGATIVE.
Respond only with a JSON object where the keys are the review numbers and the values are the classification.
USER PROMPT
0 -> An absolute masterpiece, the best film of the year!
1 -> I was bored from start to finish. A total waste of time.
2 -> The acting was incredible and the story was so moving.
3 -> While not perfect, it had some good moments.
4 -> A confusing plot and terrible dialogue.



In [6]:
from litellm import completion

response = completion(
    model = OPENAI_MODEL,
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)

print("\n model response")
print(response.choices[0].message.content)


 model response
{
  "0": "POSITIVE",
  "1": "NEGATIVE",
  "2": "POSITIVE",
  "3": "POSITIVE",
  "4": "NEGATIVE"
}


In [8]:
llm_response = {
  "0": "POSITIVE",
  "1": "NEGATIVE",
  "2": "POSITIVE",
  "3": "POSITIVE",
  "4": "NEGATIVE"
}

In [7]:
def get_accuracy(response, dataset):
    correct = 0
    total = len(response)

    for entry_number, prediction in response.items():
        entry_number = int(entry_number)
        actual_label_id = dataset[entry_number]["label"]
        actual_label = id2label[actual_label_id]

        if prediction.lower() == actual_label.lower():
            correct += 1
        else:
            print(
                f"Mismatch for entry {entry_number}: predicted={prediction}, actual={actual_label}"
            )
    accuracy = correct / total * 100
    return accuracy


In [9]:
accuracy = get_accuracy(llm_response, demo_dataset)

print("accuracy: ", accuracy)

accuracy:  100.0


In [11]:
# Create a string with the first two reviews as examples
example_string = """
Here are some examples:

EXAMPLE INPUT:
0 -> An absolute masterpiece, the best film of the year!
1 -> I was bored from start to finish. A total waste of time.

EXAMPLE OUTPUT:
{
  "0": "POSITIVE",
  "1": "NEGATIVE"
}
"""

# The new SYSTEM_PROMPT now includes the examples
IMPROVED_SYSTEM_PROMPT = SYSTEM_PROMPT + example_string

print("SYSTEM PROMPT:")
print(IMPROVED_SYSTEM_PROMPT)
print("\nUSER PROMPT:")
print(USER_PROMPT)

SYSTEM PROMPT:
You are a helpful assistant that classifies movie reviews as POSITIVE or NEGATIVE.
Respond only with a JSON object where the keys are the review numbers and the values are the classification.
Here are some examples:

EXAMPLE INPUT:
0 -> An absolute masterpiece, the best film of the year!
1 -> I was bored from start to finish. A total waste of time.

EXAMPLE OUTPUT:
{
  "0": "POSITIVE",
  "1": "NEGATIVE"
}


USER PROMPT:
0 -> An absolute masterpiece, the best film of the year!
1 -> I was bored from start to finish. A total waste of time.
2 -> The acting was incredible and the story was so moving.
3 -> While not perfect, it had some good moments.
4 -> A confusing plot and terrible dialogue.



In [12]:
response = completion(
    model=OPENAI_MODEL,
    messages = [
        { 'role': 'system', 'content': IMPROVED_SYSTEM_PROMPT },
        { 'role': 'user', 'content': USER_PROMPT },
    ],
)

print('\n llm response:')
print(response.choices[0].message.content)


 llm response:
{
  "0": "POSITIVE",
  "1": "NEGATIVE",
  "2": "POSITIVE",
  "3": "POSITIVE",
  "4": "NEGATIVE"
}
